## EVERY FRIDAY FROM 6 AM TO 11:50 PM

In [1]:
import sys
ROOT = "\\\\ominfstdpsd10001p-fs.int.okadamanila.com\\Casino\\Strategic Marketing\\Database Admin\\ETL\\Applications"
sys.path.append(f"{ROOT}\\okada\\")
from okada import SQL, EmailService, PromotexterService
from datetime import datetime, date, timedelta, time
from okada.queries.rating import LiveRating
import pandas as pd
import numpy as np
import glob
import os

loader = SQL("engage").get_loader()
hloader = SQL("hive").get_loader()


## PROFILE

In [2]:
query = f"""
SELECT 
* 
FROM [ENGAGEDB].[dbo].[UC_PL_Player] AS p 
LEFT JOIN [ENGAGEDB].[dbo].[UC_PL_Tier] pt 
	ON p.PlayerID = pt.PlayerID
LEFT JOIN [ENGAGEDB].[dbo].[UC_X_Tier] x
ON pt.TierID = x.TierID
"""

mp = loader.load_data(query)
# mp['name'] = mp['firstname'].str[0] + ' ' + mp['lastname'].str[0] + '.'
print(len(mp))
mp.head(2)

1021921


,PlayerID,UniversalPlayerID,FirstName,MiddleName,LastName,Suffix,PreferredName,FullName,BirthDate,PlayerVIP,...,TierPointsRequiredForPermUpgrade,DurationInDaysToUpgradePerm,TierPointsRequiredForTempUpgrade,DurationInDaysToUpgradeTemp,MinimumAge,CardTemplate,DiaplayExpiration,PrintExpirationDate,HEXColour,PropertyGroupID
0,1011873,OM000091212,RAMON,LIMSON,PERALTA,,RAMON PERALTA,RAMON PERALTA,1959-10-01,False,...,None,None,None,None,21.0,<CardTemplate>\r\n <frontTemplate>\r\n <Pr...,False,False,None,1.0
1,1011874,OM000091213,MARLENE,VILLARIAL,NACUA,,MARLENE NACUA,MS MARLENE NACUA,1950-04-27,False,...,None,None,None,None,21.0,<CardTemplate>\r\n <frontTemplate>\r\n <Pr...,False,False,None,1.0


In [3]:
mp.columns

Index(['PlayerID', 'UniversalPlayerID', 'FirstName', 'MiddleName', 'LastName',
       'Suffix', 'PreferredName', 'FullName', 'BirthDate', 'PlayerVIP',
       'Hosted', 'HexValue', 'TitleID', 'EnrollmentPropertyId', 'CreatedBy',
       'CreatedDate', 'UpdatedBy', 'UpdatedDate', 'ForeignFirst',
       'ForeignMiddle', 'ForeignLast', 'PlayerTypeID', 'ExpireDate', 'Alias',
       'Nationality', 'placeOfBirth', 'OccupationID', 'EnrollmentTypeID',
       'PlayerTierID', 'PlayerID', 'TierID', 'EarnedDate', 'ExpirationDate',
       'CreatedBy', 'CreatedDate', 'UpdatedBy', 'UpdatedDate', 'Justification',
       'PropertyGroupID', 'TierID', 'TierName', 'TierThreshold', 'AutoPromote',
       'AutoDemote', 'InvitationOnly', 'Visible', 'Active', 'SortOrder',
       'CreatedDate', 'CreatedBy', 'UpdatedDate', 'UpdatedBy', 'ATITierID',
       'BallyTierID', 'BallyTierID2', 'Level',
       'TierPointsRequiredForPermUpgrade', 'DurationInDaysToUpgradePerm',
       'TierPointsRequiredForTempUpgrade', 'Dur

## RATINGS

In [4]:
from datetime import datetime, timedelta

today = datetime.today()
# today = datetime(2026,7,24)

start = None
end = None

if today.weekday() == 4:  # Friday
    start = today.replace(
        hour=6,
        minute=0,
        second=0,
        microsecond=0
    )

    end = today.replace(
        hour=23,
        minute=50,
        second=0,
        microsecond=0
    )

if start is not None and end is not None:
    start_str = start.strftime('%Y-%m-%d %H:%M:%S')
    end_str = end.strftime('%Y-%m-%d %H:%M:%S')

    print(start_str)
    print(end_str)
else:
    print("Today is not Friday.")

Today is not Friday.


In [5]:
query = f"""
SELECT r.playerid PlayerID, UniversalPlayerID,
tg.RatingID RatingID,
tg.StartDatetime StartDate,
tg.EndDateTime EndDate, 
CASE
        WHEN r.transstation LIKE '01%' THEN 'H'
        ELSE 'M'
    END AS Zone,
td.TierPoint TP, r.TransactionDateTime TransactionDate
FROM ENGAGEDB.dbo.UC_PL_RatingTableGame tg
left join ENGAGEDB.dbo.UC_PL_Rating r
ON r.RatingID = tg.RatingID
left join ENGAGEDB.dbo.UC_PL_Player p
ON p.PlayerID = r.PlayerID 
left join ENGAGEDB.dbo.UC_PL_Tier t
ON t.PlayerID = p.PlayerID
left join ENGAGEDB.dbo.uc_x_tier tn
ON tn.TierID = t.TierID
left join ENGAGEDB.dbo.UC_PL_PointEarningDetail pd
ON pd.RatingID = r.RatingID 
left join ENGAGEDB.dbo.UC_PL_TierPointDetailActual td
on td.RatingID = r.RatingID
WHERE tg.EndDateTime >='{start}' AND tg.EndDateTime < '{end}'
and TableTheoWin > 0
and r.TransStation like '01%'
and TableRatingType ='MCASH'
and td.BucketID = 10024

UNION ALL 

SELECT r.playerid, UniversalPlayerID,  
eg.RatingID RatingID,
eg.StartDatetime StartDate,
eg.EndDateTime EndDate, 
eg.Zone Zone, 
td.TierPoint TP,  r.TransactionDateTime TransactionDate
FROM ENGAGEDB.dbo.UC_PL_RatingSlotGame eg
left join ENGAGEDB.dbo.UC_PL_Rating r
on r.RatingID = eg.RatingID
left join ENGAGEDB.dbo.UC_PL_Player p
on p.PlayerID = r.PlayerID 
left join ENGAGEDB.dbo.UC_PL_Tier t
on t.PlayerID = p.PlayerID
left join ENGAGEDB.dbo.uc_x_tier tn
on tn.TierID = t.TierID
left join ENGAGEDB.dbo.UC_PL_PointEarningDetail pd
on pd.RatingID = r.RatingID 
left join ENGAGEDB.dbo.UC_PL_TierPointDetailActual td
on td.RatingID = r.RatingID
where eg.EndDateTime >='{start}' AND eg.EndDateTime < '{end}'
and SlotTheoWin > 0
AND eg.Zone like 'H%'
and td.BucketID = 10024

"""
data = loader.load_data(query)
data = data.drop_duplicates()
print(len(data))
# data['universalplayerid'] = data['universalplayerid'].str.strip()
data

0


,PlayerID,UniversalPlayerID,RatingID,StartDate,EndDate,Zone,TP,TransactionDate


In [6]:
data.EndDate.min(), data.EndDate.max()

(nan, nan)

In [7]:
## join

data['UniversalPlayerID']=data['UniversalPlayerID'].str.strip()
mp['UniversalPlayerID']=mp['UniversalPlayerID'].str.strip()


data_agg = data.groupby(['UniversalPlayerID','PlayerID']).sum().reset_index()
df = data_agg.merge(mp[['UniversalPlayerID','FullName','TierName']], on='UniversalPlayerID', how='left')
print(len(df))
df.head()

KeyError: 'UniversalPlayerID'

## e-raffle 

In [ ]:
def compute_entries(df):

    # minimum TP
    tier_min_tp = {
        'prime': 5,
        'elite': 5,
        'premium': 5,
        'supreme': 5,
        'maharlika': 5, 
        'slot vvip':5,
    }

    # full max entries
    tier_max_entries = {
        'prime': 5,
        'elite': 10,
        'premium': 15,
        'supreme': 20,
        'maharlika': 25,
        'slot vvip': 25
        
    }

    df['TierName'] = (
        df['TierName']
        .astype(str)
        .str.lower()
        .str.strip()
    )

    # map values
    df['Min_TP'] = df['TierName'].map(tier_min_tp)
    df['Max_Entries'] = df['TierName'].map(tier_max_entries)

    df['Entries'] = df['TP'] / df['Min_TP']

    df['Entries'] = (
        df['Entries']
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )


    # total entries
    # double only if:
    # entries > 0 and multiplier == 1


    df['Total_Entries'] = np.minimum(
        df['Entries'],
        df['Max_Entries']
    )

    cols = ['Entries', 'Total_Entries']

    for col in cols:
        df[col] = (
            df[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
            .astype(int)
        )

    return df


daily = compute_entries(df)
print(len(daily))
daily.head(2)

In [ ]:
daily['TierName'] = np.where(
    daily['TierName'].str.contains('OKADA VIP', case=False, na=False),
    'Okada VIP',
    np.where(
        daily['TierName'].str.contains('SLOT VVIP', case=False, na=False),
        ',aharlika',
        daily['TierName']
    )
)

print(len(daily))
daily.head(2)

In [ ]:
raffle = daily.copy()
raffle = (
    raffle.groupby(['UniversalPlayerID', 'FullName'])
          .agg({'Total_Entries': 'sum'})
          .reset_index()
)

print(len(raffle))
raffle.head()

In [ ]:
raffle.UniversalPlayerID.nunique()

In [ ]:
raffle['FullName'] = raffle['FullName'].str.replace(
    r'^(Mr\.?|Ms\.?|Mrs\.?)\s+',
    '',
    regex=True,
    case=False
)

print(len(raffle))
raffle.head()

In [ ]:
daily[daily['Entries']>=5]

In [ ]:
daily[daily['UniversalPlayerID']=='OM000086492']

In [ ]:
daily.TierName.unique()

In [ ]:
def mask_name(name):
    words = str(name).split()
    
    if len(words) >= 2:
        return words[0][0] + '.' + words[-1][0] + '.'
    elif len(words) == 1:
        return words[0][0]
    else:
        return ''

raffle['Name'] = raffle['FullName'].apply(mask_name).str.upper()
print(len(raffle))
raffle.head(2)

In [ ]:
df = raffle[['UniversalPlayerID','Name','Total_Entries']]
df = df.rename(columns={'Total_Entries':'Entries'})
df = df[df['Entries']>0]
print(len(df))
df.head()

In [ ]:
df.UniversalPlayerID.nunique()

## SAVING

In [ ]:
import datetime as dt

In [ ]:
today = datetime.today()
today_str = today.strftime('%Y-%m-%d')
today_str

In [ ]:
emaildate = dt.datetime.today()
emaildate_str = emaildate.strftime('%Y-%m-%d %I %p')
emaildate, emaildate_str

In [ ]:

from datetime import datetime

now = datetime.now()

is_friday = now.strftime('%A') == 'Friday'
is_850pm = now.time() >= time(20, 50)

if is_friday and is_850pm:

    outpath1 = r'\\ominfstdpsd10001p-fs.int.okadamanila.com\Casino\Strategic Marketing\Shared Folder\P&E\Snowball Mania Fridays\E-raffle'
    outpath2 = r'\\ominfstdpsd10001p-fs.int.okadamanila.com\Casino\Casino Marketing\CRM Reports\CM List\Snowball Mania Fridays'

    df.to_csv(outpath1 + '\\' + f"Snowball Mania Fridays as of {emaildate_str}.csv",index=False) 
    df.to_csv(outpath2 + '\\' + f"Snowball Mania Fridays as of {emaildate_str}.csv",index=False) 
    
    # FOR ME
    df.to_csv(outpath3 + '\\' + f"Snowball Mania Fridays as of {emaildate_str}.csv",index=False) 
    sm.to_csv(outpath3 + '\\' + f"Snowball Mania Fridays as of {emaildate_str}.csv",index=False) 
    
    from okada import EmailService
    from okada.contacts import GAMING_ANALYTICS, 

#     recipients = f"<Neil.Sanchez@okadamanila.com>; <Dave.Valmonte@okadamanila.com>;<vanessa.aceret@okadamanila.com>;<joe.sam@okadamanila.com>;<estee.sang@okadamanila.com>; <DeborahGrace.Barcelona@okadamanila.com> ; <nino.concepcion@okadamanila.com>; <angela.valdez>"
#     copys = f"{GAMING_ANALYTICS}"

    recipients = "mary.lazatin@okadamanila.com"
    copys = "rinzai.terrado@okadamanila.com"

    mailer = EmailService(recipients,copys,f"Snowball Mania Fridays")
    mailer.send_mail(
    f"""
    <p>Kindly see the link below for the entries of Snowball Mania Fridays - {emaildate_str}.</p>
    <p>P&E Path: <a href="{outpath1}" target="_blank">{outpath1}</a></p>
    <p>PO Path: <a href="{outpath2}" target="_blank">{outpath2}</a></p>
    <p>Filename: Entries of Snowball Mania Fridays - {emaildate_str} ({len(df)}) </p>
    """)

    pass